# Requirements, Imports, and Variables

This section installs the necessary packages, sets up all imports, and defines global constants used throughout the notebook.

- **`yt-dlp`** — used to download videos from YouTube (ActivityNet, YouCook2 sources).
- **`qwen-vl-utils`** — Qwen2-VL helper that converts video/image files into the tensor format expected by the processor.
- **`trl`** — provides `SFTTrainer` and `SFTConfig`, which handle supervised fine-tuning with completion-only loss masking.

Key constants:
- `MODEL_ID` — base checkpoint (`Qwen/Qwen2-VL-2B-Instruct`).
- `FRAME_FPS = 0.2` — uniform frame rate (1 frame every 5 s) used during video pre-processing.
- `MAX_PIXELS = 256×256` — hard cap on per-frame resolution to protect GPU VRAM.
- `TOTAL_SAMPLES = 500` — total QA pairs budget that keeps a full Colab T4 run feasible end-to-end.


In [ ]:
# !pip install yt-dlp
# !pip install qwen-vl-utils
# !pip install trl

In [ ]:
# ── 1. IMPORTS ───────────────────────────────────────────────
import os, gc, json, random, subprocess, time, re
from pathlib import Path
from typing import Optional

import torch
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset
from PIL import Image
# Transformers / PEFT / TRL
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTConfig, SFTTrainer

# Qwen video utilities
from qwen_vl_utils import process_vision_info

In [ ]:
# ── 2. REPRODUCIBILITY ───────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── 3. CONSTANTS ─────────────────────────────────────────────
MODEL_ID        = "Qwen/Qwen2-VL-2B-Instruct"
OUTPUT_DIR      = "/content/qwen2vl_qlora_output"
RAW_VIDEO_DIR   = "/content/raw_videos"
PROC_VIDEO_DIR  = "/content/processed_videos"
ADAPTER_DIR     = "/content/qwen2vl_adapter"
MERGED_DIR      = "/content/qwen2vl_merged"

# Data budget: keep total small so one Colab run finishes end-to-end
TOTAL_SAMPLES   = 500   # total QA pairs to download & use
TRAIN_RATIO     = 0.80  # 400 train / 100 test
TARGET_TOPICS   = ["temporality", "event", "action"]  # per-topic eval

# Frame sampling: 1 frame every N seconds of the trimmed clip
FRAME_FPS       = 0.2
MAX_PIXELS      = 256 * 256   # hard cap per frame to protect VRAM



# Dataset Sampling

## Overview

The [VideoMarathon](https://huggingface.co/datasets/jylins/videomarathon) dataset contains QA pairs drawn from five video sources. This section describes how we filtered, weighted, and split that data to build clean train / eval / test splits suited to the task of temporal video understanding.

## Sampling Pipeline

### Step 1 — Source filtering

Two of the five sources were dropped immediately:

- **Ego4D** and **MovieChat-1K** — the video URLs in the dataset metadata are not publicly accessible, making download impossible.
- **Panda-70M** — a duration probe on 7 random videos showed a median of ~45 min and 0 % of clips under 15 min (see distribution below). Processing hour-long videos would exhaust both VRAM and Colab compute time, so this source was excluded.

The remaining sources, **ActivityNet** and **YouCook2**, have median durations of 3–5 min and are fully downloadable via `yt-dlp`.

### Step 2 — Question-type selection and weighting

The task objective is *"what happened and in what order?"*, so only question types that test temporal or sequential reasoning were retained. Each type was assigned a target count that reflects its relevance and the signal density of multiple-choice (mc) vs. open-ended (oe) formats:

| Question type | Weight (target count) |
|---|---|
| `temporality/temporal-order/mc` | 45 |
| `event/event-sequencing/mc` | 45 |
| `action/action-sequence/mc` | 45 |
| `event/key-events-extraction/mc` | 35 |
| `temporality/temporal-order/oe` | 25 |
| `event/event-sequencing/oe` | 25 |
| `action/action-sequence/oe` | 25 |
| `scene/scene-transition/mc` | 25 |

MC questions are weighted higher than OE because they provide a cleaner supervised signal (exact-match loss) and are faster to evaluate; OE questions are retained to encourage generative diversity in the fine-tuned model.

### Step 3 — Video-level train / eval / test split

Splits are performed **at the video level** (not the QA-pair level) to prevent data leakage: no video appears in more than one split. For each source separately, videos are shuffled and partitioned 70 / 15 / 15 (train / eval / test). Disjointness is asserted before proceeding.

### Step 4 — Availability verification

A fraction of YouTube videos are no longer publicly accessible. To account for this:

1. `n × 1.2` candidate videos are selected (20 % oversample buffer).
2. `yt-dlp` metadata is fetched for each candidate to confirm the video still exists and is downloadable.
3. Unavailable videos are dropped, leaving a verified pool for each split.

### Step 5 — Weighted balanced sampling

From the verified pools, QA pairs are sampled using a greedy scoring algorithm that simultaneously satisfies:

- **Question-type quotas** — samples are drawn proportional to the weights above, with remaining quota tracked per type.
- **Source balance** — equal numbers of videos and samples are drawn from ActivityNet and YouCook2.
- **Per-video caps** — at most `MAX_PER_QTYPE_PER_VIDEO = 3` questions of each type, and `MAX_QUESTIONS_PER_VIDEO = 12` questions in total, per video. This prevents any single video from dominating the dataset.

Final split sizes: **270 train / 80 eval / 80 test** QA pairs.

## Video Duration Distribution (Probe on 7 Samples per Source)

```
ActivityNet  (n=7)
  min    : 3.1 min  |  median : 3.6 min  |  mean : 3.6 min
  90th % : 4.0 min  |  max    : 4.0 min  |  <15 min: 100%  |  failed: 3

Panda-70M  (n=7)
  min    : 27.7 min  |  median : 44.7 min  |  mean : 40.4 min
  90th % : 48.5 min  |  max    : 49.4 min  |  <15 min:   0%  |  failed: 1

YouCook2  (n=7)
  min    : 3.0 min  |  median : 5.2 min  |  mean : 5.4 min
  90th % : 6.7 min  |  max    : 7.3 min  |  <15 min: 100%  |  failed: 1
```

Panda-70M's duration profile (0 % under 15 min) was the decisive factor in excluding it.


## Initial Filtering

Load the full VideoMarathon dataset and inspect its schema. The `question_type` column is the primary filter key; the `data_source` column controls which YouTube sources remain in scope.


In [ ]:
import pandas as pd
from datasets import load_dataset
df = load_dataset("jylins/videomarathon", split="train").to_pandas()
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.31k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/483M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3303740 [00:00<?, ? examples/s]

,video,data_source,answer,question_type,id,question,URL
0,activitynet/v_-vDMeHr1ZfI.mp4,ActivityNet,The video captures the transition from day to ...,temporality/temporal-perception/oe,videomarathon_00414943,What is the focus of the video in the first 30...,https://www.youtube.com/watch?v=-vDMeHr1ZfI
1,activitynet/v_-vDMeHr1ZfI.mp4,ActivityNet,The video shifts from a static sunset scene at...,temporality/temporal-perception/oe,videomarathon_00414944,How does the video transition from the restaur...,https://www.youtube.com/watch?v=-vDMeHr1ZfI
2,activitynet/v_-vDMeHr1ZfI.mp4,ActivityNet,The final scene shows a bustling conveyor belt...,temporality/temporal-perception/oe,videomarathon_00414945,What does the final scene of the video depict?,https://www.youtube.com/watch?v=-vDMeHr1ZfI
3,activitynet/v_-vDMeHr1ZfI.mp4,ActivityNet,The kayaking adventure is shown in more clips ...,temporality/duration-perception/oe,videomarathon_00414946,"Which activity is shown in more clips, the kay...",https://www.youtube.com/watch?v=-vDMeHr1ZfI
4,activitynet/v_-vDMeHr1ZfI.mp4,ActivityNet,"No, the transition from the river to the sushi...",temporality/duration-perception/oe,videomarathon_00414947,Does the transition from the serene river to t...,https://www.youtube.com/watch?v=-vDMeHr1ZfI


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3303740 entries, 0 to 3303739
Data columns (total 7 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   video          object
 1   data_source    object
 2   answer         object
 3   question_type  object
 4   id             object
 5   question       object
 6   URL            object
dtypes: object(7)
memory usage: 176.4+ MB


In [ ]:
df.iloc[0].values

array(['activitynet/v_-vDMeHr1ZfI.mp4', 'ActivityNet',
       'The video captures the transition from day to night, highlighting the serene ambiance of a restaurant by the water.',
       'temporality/temporal-perception/oe', 'videomarathon_00414943',
       'What is the focus of the video in the first 30 seconds?',
       'https://www.youtube.com/watch?v=-vDMeHr1ZfI'], dtype=object)

In [ ]:
QUESTION_TYPES = {
    'temporality/temporal-order/mc'  : 45,
    'temporality/temporal-order/oe'  : 25,
    'event/event-sequencing/mc'      : 45,
    'event/event-sequencing/oe'      : 25,
    'action/action-sequence/mc'      : 45,
    'action/action-sequence/oe'      : 25,
    'event/key-events-extraction/mc' : 35,
    'scene/scene-transition/mc'      : 25,
}
QTYPES = list(QUESTION_TYPES.keys())

In [ ]:
dft = df[df["question_type"].isin(QTYPES)]
dft["data_source"].value_counts()

,count
data_source,
Panda-70M,590284
Ego4D,201646
ActivityNet,77461
YouCook2,52261
MovieChat-1K,25393


In [ ]:
# moviechat-1k and Ego4D are removed as the
df.groupby("data_source")["video"].nunique()

,video
data_source,
ActivityNet,2293
Panda-70M,17476
YouCook2,1551


## Identifying Videos Available for Download

Not all videos listed in the dataset are still accessible on YouTube. This step:

1. Computes per-source candidate pools with a **1.2× oversample factor** to absorb expected attrition.
2. Prioritises videos that cover more question types (`qt_coverage`) and have more QA pairs (`n_questions`), maximising the diversity of each video downloaded.
3. Calls `get_youtube_duration()` (a lightweight `yt-dlp` metadata fetch — no full download) to verify each candidate is reachable and within the duration budget.
4. Returns the verified subset as the pool from which QA pairs are finally sampled.


In [ ]:
def split_videos_by_source(df, seed=42):
    train_vids, eval_vids, test_vids = set(), set(), set()
    rng = np.random.default_rng(seed)

    for source in SOURCES:
        vids = df.loc[df["data_source"] == source, "video"].unique()
        rng.shuffle(vids)

        n = len(vids)
        n_test = max(1, int(0.15 * n))
        n_eval = max(1, int(0.15 * n))

        test_vids.update(vids[:n_test])
        eval_vids.update(vids[n_test:n_test + n_eval])
        train_vids.update(vids[n_test + n_eval:])

    assert not (train_vids & eval_vids)
    assert not (train_vids & test_vids)
    assert not (eval_vids & test_vids)

    return train_vids, eval_vids, test_vids

    
train_vids, eval_vids, test_vids = split_videos_by_source(df)

In [ ]:
def get_candidate_videos(
    df,
    video_caps,
    oversample_factor=1.2,
    seed=42
):
    rng = np.random.default_rng(seed)
    candidates = []

    video_meta = (
        df.groupby("video")
          .agg(
              data_source=("data_source", "first"),
              URL=("URL", "first"),
              qt_coverage=("question_type", "nunique"),
              n_questions=("question_type", "size"),
          )
          .reset_index()
    )

    for source, cap in video_caps.items():
        pool = video_meta[video_meta["data_source"] == source].copy()

        # prefer videos with more question types/questions
        pool = pool.sort_values(
            ["qt_coverage", "n_questions"],
            ascending=False
        )

        n_candidates = min(len(pool), int(floor(cap * oversample_factor)))

        top_pool = pool.head(n_candidates * 2)

        chosen = top_pool.sample(
            n=n_candidates,
            random_state=seed
        )

        candidates.append(chosen)

    return pd.concat(candidates).reset_index(drop=True)

In [ ]:
from tqdm import tqdm
def verify_candidate_videos(
    candidate_videos,
):
    verified = []

    for _, row in tqdm(candidate_videos.iterrows(), total=len(candidate_videos)):
        duration = get_youtube_duration(row["URL"])
        print(duration)
        verified.append({
            **row.to_dict(),
            "duration": duration,
            "exists": duration is not None,
            "under_limit": duration,
        })


    verified = pd.DataFrame(verified)

    valid_videos = verified[
        verified["exists"]
        & verified["under_limit"]
    ]["video"].tolist()

    return valid_videos, verified

In [ ]:
TRAIN_VIDEO_CAPS = {
    "ActivityNet": 40,
    # "Panda-70M": 30,
    "YouCook2": 40,
}

EVAL_TEST_VIDEO_CAPS = {
    "ActivityNet": 10,
    # "Panda-70M": 10,
    "YouCook2": 10,
}


In [ ]:

df_train_pool = df[df["video"].isin(train_vids)]
df_eval_pool  = df[df["video"].isin(eval_vids)]
df_test_pool  = df[df["video"].isin(test_vids)]

print("\n── Verifying TRAIN candidate videos ──")
train_candidates = get_candidate_videos(
    df_train_pool,
    TRAIN_VIDEO_CAPS,
    oversample_factor=1.2,
    seed=42
)

In [ ]:
train_candidates

,video,data_source,URL,qt_coverage,n_questions
0,activitynet/v_2WKy0FvMtCM.mp4,ActivityNet,https://www.youtube.com/watch?v=2WKy0FvMtCM,8,19
1,activitynet/v_2Mj26IwwEiY.mp4,ActivityNet,https://www.youtube.com/watch?v=2Mj26IwwEiY,8,19
2,activitynet/v_2AjyB3mCW_U.mp4,ActivityNet,https://www.youtube.com/watch?v=2AjyB3mCW_U,8,19
3,activitynet/v_2xgecBn6YwM.mp4,ActivityNet,https://www.youtube.com/watch?v=2xgecBn6YwM,8,19
4,activitynet/v_0GpNcvAVWVg.mp4,ActivityNet,https://www.youtube.com/watch?v=0GpNcvAVWVg,8,19
...,...,...,...,...,...
91,youcook2/0hVAuvPAIzA.mp4,YouCook2,https://www.youtube.com/watch?v=0hVAuvPAIzA,8,19
92,youcook2/3sEVxqIaxfc.mp4,YouCook2,https://www.youtube.com/watch?v=3sEVxqIaxfc,8,19
93,youcook2/0xYOUNfzIn8.mp4,YouCook2,https://www.youtube.com/watch?v=0xYOUNfzIn8,8,19
94,youcook2/4nxbRG6-sfw.mp4,YouCook2,https://www.youtube.com/watch?v=4nxbRG6-sfw,8,19


In [ ]:
train_valid_videos, train_verified = verify_candidate_videos(
    train_candidates
)

  1%|          | 1/96 [00:03<06:14,  3.94s/it]

192.0


  2%|▏         | 2/96 [00:06<04:59,  3.19s/it]

  ⚠️  yt-dlp error: WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction
None


  3%|▎         | 3/96 [00:09<04:36,  2.97s/it]

...

579.0


 99%|█████████▉| 95/96 [04:04<00:02,  2.42s/it]

345.0


100%|██████████| 96/96 [04:06<00:00,  2.57s/it]

  ⚠️  yt-dlp error: WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction
None


In [ ]:
len(train_candidates),len(train_valid_videos), len(train_verified)

(96, 78, 96)

In [ ]:
# df_train_pool = df[df["video"].isin(train_vids)]
df_eval_pool  = df[df["video"].isin(eval_vids)]
df_test_pool  = df[df["video"].isin(test_vids)]

print("\n── Verifying EVAL candidate videos ──")
eval_candidates = get_candidate_videos(
    df_eval_pool,
    EVAL_TEST_VIDEO_CAPS,
    oversample_factor=1.2,
    seed=43
)

eval_valid_videos, eval_verified = verify_candidate_videos(
    eval_candidates
)

df_eval_pool = df_eval_pool[df_eval_pool["video"].isin(eval_valid_videos)]


print("\n── Verifying TEST candidate videos ──")
test_candidates = get_candidate_videos(
    df_test_pool,
    EVAL_TEST_VIDEO_CAPS,
    oversample_factor=1.2,
    seed=44
)

test_valid_videos, test_verified = verify_candidate_videos(
    test_candidates
)

df_test_pool = df_test_pool[df_test_pool["video"].isin(test_valid_videos)]


── Verifying EVAL candidate videos ──


  4%|▍         | 1/24 [00:04<01:45,  4.59s/it]

236.0


  8%|▊         | 2/24 [00:07<01:23,  3.82s/it]

229.0


...

244.0


 96%|█████████▌| 23/24 [01:14<00:02,  2.61s/it]

369.0


100%|██████████| 24/24 [01:16<00:00,  3.20s/it]

197.0


## Sampling from the Verified Video Pool

With confirmed-available videos in hand, this step draws the final QA pairs for each split using `sample_weighted_balanced`. The algorithm:

- Iterates over candidate videos ranked by a composite score that rewards coverage of still-unfilled question-type quotas and under-represented sources.
- For each selected video, takes up to `MAX_PER_QTYPE_PER_VIDEO = 3` rows per question type and `MAX_QUESTIONS_PER_VIDEO = 12` rows in total.
- Deducts filled quota counts at each step so later iterations focus on remaining gaps.
- Stops when all quotas are satisfied or no more eligible videos remain.

This greedy approach is deterministic (seeded), reproducible, and avoids the imbalance that a naive random sample would introduce given the unequal frequency of question types in the raw dataset.


In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

QUESTION_TYPES = {
    'temporality/temporal-order/mc'  : 45,
    'temporality/temporal-order/oe'  : 25,
    'event/event-sequencing/mc'      : 45,
    'event/event-sequencing/oe'      : 25,
    'action/action-sequence/mc'      : 45,
    'action/action-sequence/oe'      : 25,
    'event/key-events-extraction/mc' : 35,
    'scene/scene-transition/mc'      : 25,
}

SOURCES = ["ActivityNet", "YouCook2"]
QTYPES = list(QUESTION_TYPES.keys())

TRAIN_TOTAL = 270
EVAL_TOTAL  = 80
TEST_TOTAL  = 80

MAX_PER_QTYPE_PER_VIDEO = 3
MAX_QUESTIONS_PER_VIDEO = 12

TRAIN_VIDEO_CAPS = {
    "ActivityNet": 40,
    # "Panda-70M": 30,
    "YouCook2": 40,
}

EVAL_TEST_VIDEO_CAPS = {
    "ActivityNet": 10,
    # "Panda-70M": 10,
    "YouCook2": 10,
}


def make_weighted_targets(weights, total):
    weight_sum = sum(weights.values())
    raw = {k: total * v / weight_sum for k, v in weights.items()}

    targets = {k: int(np.floor(v)) for k, v in raw.items()}
    leftover = total - sum(targets.values())

    remainders = sorted(
        raw.items(),
        key=lambda x: x[1] - np.floor(x[1]),
        reverse=True
    )

    for k, _ in remainders[:leftover]:
        targets[k] += 1

    return targets


TRAIN_QT_TARGETS = make_weighted_targets(QUESTION_TYPES, TRAIN_TOTAL)
EVAL_QT_TARGETS  = make_weighted_targets(QUESTION_TYPES, EVAL_TOTAL)
TEST_QT_TARGETS  = make_weighted_targets(QUESTION_TYPES, TEST_TOTAL)


def sample_weighted_balanced(
    df,
    qt_targets,
    source_video_caps,
    seed=42,
    max_per_qtype_per_video=3,
    max_questions_per_video=12,
):
    rng = np.random.default_rng(seed)

    df = df[
        df["data_source"].isin(SOURCES)
        & df["question_type"].isin(qt_targets.keys())
    ].copy()

    total_target = sum(qt_targets.values())

    source_sample_targets = {
        s: total_target // len(SOURCES)
        for s in SOURCES
    }

    # distribute remainder
    for s in SOURCES[: total_target % len(SOURCES)]:
        source_sample_targets[s] += 1

    video_groups = {
        vid: g
        for vid, g in df.groupby("video", sort=False)
    }

    video_meta = (
        df.groupby("video")
          .agg(
              data_source=("data_source", "first"),
              qt_coverage=("question_type", lambda x: len(set(x) & set(qt_targets))),
              n_questions=("question_type", "size"),
          )
          .reset_index()
    )

    remaining_qt = qt_targets.copy()
    source_video_counts = defaultdict(int)
    source_sample_counts = defaultdict(int)

    selected_rows = []
    selected_vids = set()

    def video_score(vrow):
        vid = vrow["video"]
        g = video_groups[vid]

        score = 0.0
        for qt in g["question_type"].unique():
            if remaining_qt.get(qt, 0) > 0:
                # higher weight question types get preferred
                score += remaining_qt[qt] / qt_targets[qt]

        source = vrow["data_source"]

        # prefer under-covered sources
        source_need = max(
            0,
            source_sample_targets[source] - source_sample_counts[source]
        )

        return score + 0.5 * source_need / max(1, source_sample_targets[source])

    while any(v > 0 for v in remaining_qt.values()):
        candidates = video_meta[
            ~video_meta["video"].isin(selected_vids)
        ].copy()

        candidates = candidates[
            candidates["data_source"].map(source_video_counts)
            < candidates["data_source"].map(source_video_caps)
        ]

        if candidates.empty:
            break

        candidates["score"] = candidates.apply(video_score, axis=1)
        candidates = candidates[candidates["score"] > 0]

        if candidates.empty:
            break

        candidates = candidates.sort_values("score", ascending=False)

        picked = False

        for _, vrow in candidates.iterrows():
            vid = vrow["video"]
            source = vrow["data_source"]
            g = video_groups[vid]

            rows_for_video = []

            # sample rare / high-remaining qtypes first
            qtypes_in_video = list(g["question_type"].unique())
            qtypes_in_video.sort(
                key=lambda qt: remaining_qt.get(qt, 0) / max(1, qt_targets.get(qt, 1)),
                reverse=True
            )

            video_taken = 0

            for qt in qtypes_in_video:
                if remaining_qt.get(qt, 0) <= 0:
                    continue

                if video_taken >= max_questions_per_video:
                    break

                qt_rows = g[g["question_type"] == qt]

                source_room = max(
                    0,
                    source_sample_targets[source] - source_sample_counts[source]
                )

                # allow slight overflow only if qtype quota still needs filling
                source_limit = max(1, source_room)

                n_take = min(
                    len(qt_rows),
                    remaining_qt[qt],
                    max_per_qtype_per_video,
                    max_questions_per_video - video_taken,
                    source_limit,
                )

                if n_take <= 0:
                    continue

                rows_for_video.append(
                    qt_rows.sample(
                        n=n_take,
                        random_state=seed + len(selected_vids)
                    )
                )

                remaining_qt[qt] -= n_take
                source_sample_counts[source] += n_take
                video_taken += n_take

            if rows_for_video:
                selected_rows.extend(rows_for_video)
                selected_vids.add(vid)
                source_video_counts[source] += 1
                picked = True
                break

        if not picked:
            break

    if not selected_rows:
        return pd.DataFrame(columns=df.columns)

    df_out = pd.concat(selected_rows).reset_index(drop=True)

    print("=" * 70)
    print(f"Total samples : {len(df_out)} / {total_target}")
    print(f"Unique videos : {df_out['video'].nunique()}")

    print("\nVideos per source:")
    print(df_out.groupby("data_source")["video"].nunique().to_string())

    print("\nSamples per source:")
    print(df_out["data_source"].value_counts().to_string())

    print("\nQuestion type distribution:")
    actual = df_out["question_type"].value_counts()
    for qt, target in qt_targets.items():
        got = actual.get(qt, 0)
        status = "✅" if got >= target else "⚠️"
        print(f"{status} {qt:45s}: {got:3d} / {target}")

    unmet = {qt: v for qt, v in remaining_qt.items() if v > 0}
    if unmet:
        print("\n⚠️ Unmet question-type quotas:")
        print(unmet)

    return df_out





In [ ]:

def full_pipeline(df_train_pool, df_eval_pool, df_test_pool):

    print("Pool sizes:")
    print("train:", df_train_pool["video"].nunique())
    print("eval :", df_eval_pool["video"].nunique())
    print("test :", df_test_pool["video"].nunique())

    print("\n── Sampling TRAIN ──")
    df_train = sample_weighted_balanced(
        df_train_pool,
        TRAIN_QT_TARGETS,
        TRAIN_VIDEO_CAPS,
        seed=42,
    )

    print("\n── Sampling EVAL ──")
    df_eval = sample_weighted_balanced(
        df_eval_pool,
        EVAL_QT_TARGETS,
        EVAL_TEST_VIDEO_CAPS,
        seed=43,
    )

    print("\n── Sampling TEST ──")
    df_test = sample_weighted_balanced(
        df_test_pool,
        TEST_QT_TARGETS,
        EVAL_TEST_VIDEO_CAPS,
        seed=44,
    )

    print("\n" + "=" * 70)
    print("FINAL DATASET SUMMARY")
    print("=" * 70)

    for name, split in [
        ("TRAIN", df_train),
        ("EVAL", df_eval),
        ("TEST", df_test),
    ]:
        print(f"\n{name}")
        print("samples:", len(split))
        print("videos :", split["video"].nunique())
        print("sources:")
        print(split["data_source"].value_counts().to_string())

    total_videos = (
        set(df_train["video"])
        | set(df_eval["video"])
        | set(df_test["video"])
    )

    print(f"\nTotal unique videos to download: {len(total_videos)}")
    print(f"Estimated download time T4: {len(total_videos) * 3 / 60:.1f} hrs")

    df_train.to_csv("/content/train_split_LOCKED.csv", index=False)
    df_eval.to_csv("/content/eval_split_LOCKED.csv", index=False)
    df_test.to_csv("/content/test_split_LOCKED.csv", index=False)

    return df_train, df_eval, df_test
df_train, df_eval, df_test = full_pipeline(df_train_pool,df_eval_pool, df_test_pool)

Pool sizes:
train: 78
eval : 20
test : 22

── Sampling TRAIN ──
Total samples : 270 / 270
Unique videos : 31

Videos per source:
data_source
ActivityNet    23
YouCook2        8

Samples per source:
data_source
ActivityNet    176
YouCook2        94

Question type distribution:
✅ temporality/temporal-order/mc                :  45 / 45
✅ temporality/temporal-order/oe                :  25 / 25
✅ event/event-sequencing/mc                    :  45 / 45
✅ event/event-sequencing/oe                    :  25 / 25
✅ action/action-sequence/mc                    :  45 / 45
✅ action/action-sequence/oe                    :  25 / 25
✅ event/key-events-extraction/mc               :  35 / 35
✅ scene/scene-transition/mc                    :  25 / 25

── Sampling EVAL ──
Total samples : 80 / 80
Unique videos : 10

Videos per source:
data_source
ActivityNet    7
YouCook2       3

Samples per source:
data_source
ActivityNet    48
YouCook2       32

Question type distribution:
✅ temporality/temporal-order/mc

# Data Preparation for Training

Raw QA rows from the sampled splits cannot be fed directly to `SFTTrainer` — each row must be converted into tokenised `input_ids` / `attention_mask` tensors that already embed the video frames. This section describes the two-phase pipeline that accomplishes this efficiently.

## Phase 1 — Build a per-video frame cache

Downloading and FFmpeg-processing the same video once per QA row would be extremely wasteful (a single video may contribute up to 12 QA pairs). Instead, **each unique video is processed exactly once**:

1. **Download** — `yt-dlp` fetches the lowest-quality MP4 to minimise download time and disk usage.
2. **Frame sampling** — FFmpeg extracts frames at `FRAME_FPS = 0.2` (1 frame per 5 s) with CRF-28 compression and `ultrafast` preset, producing a compact processed video. The raw file is deleted immediately to free disk space.
3. **Tensor conversion** — `process_vision_info()` converts the processed video into a list of pixel tensors. These tensors are stored in an in-memory dictionary keyed by video ID.

Uniform temporal stride was chosen over scene-change detection because temporal-order questions require evenly spaced frame coverage; adaptive sampling could skip frames from key transitions that the questions reference.

## Phase 2 — Tokenise QA pairs from cache

For each QA row, the cached video tensors are retrieved (no re-download, no FFmpeg), a chat-formatted prompt is built using `apply_chat_template`, and the processor tokenises the combined text + video input. The resulting tensors are stored in HuggingFace `Dataset` format compatible with `SFTTrainer`.

Processed video files are deleted after Phase 2 to reclaim disk space before the next split is built.


In [ ]:
import os, gc, time, shutil
import torch
import pandas as pd
from datasets import Dataset
from typing import Optional
from qwen_vl_utils import process_vision_info
# data preperation
def build_conversation(video_path: str, question: str, answer: str) -> list[dict]:
    """
    Format one training example into Qwen2-VL's chat template.

    We use the assistant-turn answer so SFTTrainer can compute
    the cross-entropy loss only on the response tokens
    (label masking is handled by DataCollatorForCompletionOnlyLM
     or trl's built-in completion-only mode).
    """
    print("inside build conversation")
    return [
        {
            "role": "user",
            "content": [
                {"type": "video", "video": f"file://{video_path}"},
                {"type": "text",  "text": question},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": answer}],
        },
    ]


def download_video(url: str, video_id: str, output_dir: str = RAW_VIDEO_DIR) -> str:
    """Download lowest-quality MP4 with yt-dlp; preserving internal relative subfolders."""
    print("inside download_video")

    # 1. Combine the base tracking folder with the relative video path
    # e.g., /your_raw_dir/ + activitynet/v_-FWGLSfI13Q.mp4
    out_path = os.path.join(output_dir, video_id)

    # 2. Extract the nested parent folder directory structure and create it
    # e.g., extracts '/your_raw_dir/activitynet' and ensures it exists
    nested_dir = os.path.dirname(out_path)
    os.makedirs(nested_dir, exist_ok=True)

    # 3. Simple local file caching validation check
    if os.path.exists(out_path):
        print(f"⏭️  Already downloaded: {out_path}")
        return out_path

    # 4. Invoke download task directly targetting the rigid file path
    cmd = [
        "yt-dlp", "-f", "worst[ext=mp4]/worst",
        "--merge-output-format", "mp4",
        "--socket-timeout", "30",
        "-o", out_path, url,
    ]

    print(f"⬇️ Running yt-dlp target destination: {out_path} ...")
    subprocess.run(cmd, check=True, capture_output=True, timeout=120)

    # 5. Direct safety confirmation check
    if os.path.exists(out_path):
        print("download video completed")
        return out_path

    raise FileNotFoundError(f"yt-dlp completed but no file was written to target destination: {out_path}")

def trim_and_sample_frames(
    input_path: str,
    output_path: str,
    fps: float = FRAME_FPS,
    # start_sec: Optional[float] = None,
    # end_sec: Optional[float] = None,
) -> str:
    """
    FFmpeg: optionally trim to [start, end], then extract frames at `fps`.

    Frame-sampling strategy — justified here:
    ──────────────────────────────────────────
    VideoMarathon clips are long (minutes to hours).
    We use a uniform temporal stride (1 frame / 2 s) rather than
    random sampling because:
      • Temporal-order questions need evenly spaced coverage.
      • 1 frame/2 s → ~8–30 frames for a typical 16–60 s clip —
        manageable under Qwen2-VL's sequence-length budget.
      • Adaptive strategies (scene-change detection) are better
        but require a second ffprobe pass; overkill for a subset run.
    For very long clips we additionally honour start/end timestamps
    from the dataset's 'timestamp' field to avoid processing dead air.
    """
    print("inside trim and sample frames")
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    cmd = [
        "ffmpeg", "-y",
        "-i", input_path,
        "-vf", f"fps={fps}",          # Downsample frame rate
        "-vcodec", "libx264",
        "-crf", "28",                  # Aggressive compression
        "-preset", "ultrafast",
        output_path
    ]
    subprocess.run(cmd, check=True, capture_output=True, timeout=300)
    print("trim_sample_completed")
    return output_path

# ── Phase 1: Build video cache for unique videos ──────────────────────────────

def build_video_cache(
    df: pd.DataFrame,
    processor,
    raw_dir: str = RAW_VIDEO_DIR,
    proc_dir: str = PROC_VIDEO_DIR,
) -> dict:
    """
    Download + FFmpeg-sample every unique video once.
    Returns:
        cache = {
            sid: {
                "video_inputs": Tensor,   # shape [T, C, H, W] — already on CPU
                "proc_path":    str,      # kept alive so process_vision_info can re-read if needed
            }
        }
    Failures are logged and the sid is omitted from the cache;
    rows referencing it will be skipped in Phase 2.
    """
    unique_sids = df["video"].unique()
    print(f"📹 Unique videos to process: {len(unique_sids)}")

    cache = {}

    for i, sid in enumerate(unique_sids):
        print(f"  [{i+1}/{len(unique_sids)}] Processing {sid} ...", end="\r")

        # Pick any row for this sid to get the URL (URL is per-video, not per-QA)
        sample_row = df[df["video"] == sid].iloc[0]
        url = sample_row.get("URL", sample_row.get("url", ""))

        raw_path  = None
        proc_path = os.path.join(proc_dir, f"{sid}")

        try:
            # 1. Download
            raw_path = download_video(url, sid, raw_dir)

            # 2. FFmpeg frame-sample
            proc_path = trim_and_sample_frames(raw_path, proc_path, fps=FRAME_FPS)

            # 3. Run process_vision_info once using a minimal dummy conversation
            #    — we only need video_inputs here; image_inputs will be [] for video
            dummy_conversation = [
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "video",
                            "video": f"file://{proc_path}",
                            "fps": FRAME_FPS,           # suppress smart_nframes warning
                        },
                        {"type": "text", "text": "dummy"},
                    ],
                }
            ]
            _, video_inputs = process_vision_info(dummy_conversation)

            cache[sid] = {
                "video_inputs": video_inputs,   # list of tensors
                "proc_path":    proc_path,
            }
            print(f"  ✅ [{i+1}/{len(unique_sids)}] {sid} — frames: {video_inputs[0].shape}")

        except Exception as e:
            print(f"\n  ⚠️  Skipped video {sid}: {e}")

        finally:
            # Delete raw (large) file immediately; keep proc file until Phase 2 done
            if raw_path and os.path.exists(raw_path):
                os.remove(raw_path)
            gc.collect()

    print(f"\n✅ Video cache built: {len(cache)}/{len(unique_sids)} videos ready")
    return cache


# ── Phase 2: Tokenize all 270 QA rows using cached video tensors ──────────────

def prepare_sample_from_cache(
    row: pd.Series,
    processor,
    video_cache: dict,
) -> Optional[dict]:
    """
    Build one tokenised training record using pre-cached video_inputs.
    No downloading or FFmpeg — just prompt building + processor call.
    """
    sid      = row["video"]
    question = str(row.get("question", ""))
    answer   = str(row.get("answer", ""))

    if sid not in video_cache:
        print(f"  ⚠️  Skipped QA row (video not in cache): {sid}")
        return None

    cached      = video_cache[sid]
    video_inputs = cached["video_inputs"]
    proc_path   = cached["proc_path"]

    try:
        # Build conversation (still needed for apply_chat_template)
        conversation = build_conversation(proc_path, question, answer)

        prompt = processor.apply_chat_template(
            conversation, tokenize=False, add_generation_prompt=False
        )

        # Reuse cached video_inputs — skip process_vision_info entirely
        image_inputs = None   # no images in this dataset

        inputs = processor(
            text=[prompt],
            images=image_inputs,
            videos=video_inputs,        # ← from cache, not recomputed
            padding=False,
            return_tensors="pt",
            max_pixels=MAX_PIXELS,
        )

        sample = {k: v[0].tolist() for k, v in inputs.items()}
        sample["_topic"]    = row.get("_topic", "unknown")
        sample["_id"]       = sid
        sample["_question"] = question
        sample["_answer"]   = answer
        return sample

    except Exception as e:
        print(f"  ⚠️  Skipped QA {sid} | Q: {question[:40]}... → {e}")
        return None


# ── Phase 3: Assemble HF Dataset ─────────────────────────────────────────────

def build_hf_dataset_cached(
    df: pd.DataFrame,
    processor,
    split_name: str,
    proc_dir: str = PROC_VIDEO_DIR,
) -> Dataset:
    """
    Full pipeline:
      1. Build video cache (31 videos processed once)
      2. Tokenize all 270 QA rows against cache
      3. Clean up processed video files
      4. Return HF Dataset
    """
    print(f"\n{'='*55}")
    print(f"  Building [{split_name}] dataset — {len(df)} rows, "
          f"{df['video'].nunique()} unique videos")
    print(f"{'='*55}")

    # ── 1. Video cache ─────────────────────────────────────────
    video_cache = build_video_cache(df, processor, proc_dir=proc_dir)

    # ── 2. Tokenize QA rows ────────────────────────────────────
    records = []
    print(f"\n🔤 Tokenizing {len(df)} QA rows...")

    for i, (_, row) in enumerate(df.iterrows()):
        t0  = time.time()
        rec = prepare_sample_from_cache(row, processor, video_cache)
        elapsed = time.time() - t0

        status = "✅" if rec is not None else "❌"
        print(f"  {status} [{i+1}/{len(df)}] {row['video']} "
              f"({elapsed:.1f}s)")

        if rec is not None:
            records.append(rec)

    print(f"\n✅ {split_name}: {len(records)}/{len(df)} samples tokenized")

    # ── 3. Disk cleanup ────────────────────────────────────────
    print(f"🧹 Cleaning up {proc_dir} ...")
    if os.path.exists(proc_dir):
        shutil.rmtree(proc_dir)
        os.makedirs(proc_dir, exist_ok=True)
    print("✨ Cleanup complete.")

    # ── 4. Return dataset ──────────────────────────────────────
    return Dataset.from_list(records)


# ── Entry point ───────────────────────────────────────────────────────────────

In [ ]:
hf_eval = build_hf_dataset_cached(df_eval, processor, "eval")
hf_test = build_hf_dataset_cached(df_test, processor, "test")

In [ ]:
hf_train = build_hf_dataset_cached(df_train, processor, "train")


  Building [train] dataset — 270 rows, 31 unique videos
📹 Unique videos to process: 31
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/activitynet/v_-FWGLSfI13Q.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [1/31] activitynet/v_-FWGLSfI13Q.mp4 — frames: torch.Size([44, 3, 364, 644])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/youcook2/38STPyrFTug.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [2/31] youcook2/38STPyrFTug.mp4 — frames: torch.Size([58, 3, 364, 644])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/activitynet/v_-KbDXeEoQ1E.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [3/31] activitynet/v_-KbDXeEoQ1E.mp4 — frames: torch.Size([40, 3, 364, 644])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/youcook2/3V4MxH2GuIU.mp4 .

In [ ]:
hf_train.to_pandas()

,input_ids,attention_mask,mm_token_type_ids,pixel_values_videos,video_grid_thw,_topic,_id,_question,_answer
0,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.15723881125450134, -0.15723881125450134, -...","[22, 26, 46]",unknown,activitynet/v_-FWGLSfI13Q.mp4,What was the order of events in the dimly lit ...,"In the dimly lit room, the child first explore..."
1,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.15723881125450134, -0.15723881125450134, -...","[22, 26, 46]",unknown,activitynet/v_-FWGLSfI13Q.mp4,Describe the sequence of events from the start...,The video starts with a pumpkin carving sessio...
2,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.15723881125450134, -0.15723881125450134, -...","[22, 26, 46]",unknown,activitynet/v_-FWGLSfI13Q.mp4,What was the sequence of activities related to...,The sequence included the initial carving by t...
3,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.15723881125450134, -0.15723881125450134, -...","[22, 26, 46]",unknown,activitynet/v_-FWGLSfI13Q.mp4,"After the family finishes carving the pumpkin,...",The scene transitions to a dimly lit room wher...
4,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-0.15723881125450134, -0.15723881125450134, -...","[22, 26, 46]",unknown,activitynet/v_-FWGLSfI13Q.mp4,After the man helps the child carve the pumpki...,The man and children proceed to cut open the p...
...,...,...,...,...,...,...,...,...,...
255,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-1.6024830341339111, -1.6024830341339111, -1....","[22, 22, 40]",unknown,activitynet/v_0eR57IP6paM.mp4,What comes first in the sequence of actions: a...,D. Explaining ski pole usage
256,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-1.6024830341339111, -1.6024830341339111, -1....","[22, 22, 40]",unknown,activitynet/v_0eR57IP6paM.mp4,What is the correct chronological order of act...,"B. Cleaning the skis, demonstrating ski pole u..."
257,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-1.7922625541687012, -1.7922625541687012, -1....","[23, 18, 30]",unknown,activitynet/v_0hdwFR5qWz4.mp4,What is the correct sequence of events in the ...,C.
258,"[151644, 8948, 198, 2610, 525, 264, 10950, 178...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[-1.7922625541687012, -1.7922625541687012, -1....","[23, 18, 30]",unknown,activitynet/v_0hdwFR5qWz4.mp4,Which action occurs first in the video?\nOptio...,A. The camera pans across the cityscape and ri...


In [ ]:
hf_test = build_hf_dataset_cached(df_test, processor,"eval")


  Building [eval] dataset — 80 rows, 10 unique videos
📹 Unique videos to process: 10
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/activitynet/v_-76d-7Ju7L0.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [1/10] activitynet/v_-76d-7Ju7L0.mp4 — frames: torch.Size([38, 3, 364, 476])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/youcook2/2SxbO4VAgN8.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [2/10] youcook2/2SxbO4VAgN8.mp4 — frames: torch.Size([82, 3, 364, 644])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/activitynet/v_-Q03gEypilg.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [3/10] activitynet/v_-Q03gEypilg.mp4 — frames: torch.Size([40, 3, 364, 476])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/youcook2/-h8c5sLelZ4.mp4 ...

In [ ]:
hf_eval = build_hf_dataset_cached(df_eval, processor, "eval")



  Building [eval] dataset — 80 rows, 10 unique videos
📹 Unique videos to process: 10
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/activitynet/v_-wOaPhSf6OE.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [1/10] activitynet/v_-wOaPhSf6OE.mp4 — frames: torch.Size([40, 3, 364, 476])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/youcook2/1mB0G1AwUPg.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [2/10] youcook2/1mB0G1AwUPg.mp4 — frames: torch.Size([62, 3, 364, 644])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/activitynet/v_0_1BQPWzRiw.mp4 ...
download video completed
inside trim and sample frames
trim_sample_completed
  ✅ [3/10] activitynet/v_0_1BQPWzRiw.mp4 — frames: torch.Size([36, 3, 364, 644])
inside download_video
⬇️ Running yt-dlp target destination: /content/raw_videos/youcook2/41NQ9UXkPWE.mp4 ...

# Training

## Approach: QLoRA Supervised Fine-Tuning on Qwen2-VL-2B

The base model is fine-tuned using **QLoRA** (Quantised Low-Rank Adaptation) — a parameter-efficient method that keeps the base weights frozen and quantised in 4-bit NF4 while training a small set of low-rank adapter matrices. This makes it possible to fine-tune a 2B-parameter vision-language model on a single T4 GPU (15 GB VRAM).

### Pipeline

1. Load `Qwen/Qwen2-VL-2B-Instruct` in 4-bit NF4 quantisation via `BitsAndBytesConfig`.
2. Call `prepare_model_for_kbit_training()` to stabilise gradient flow through quantised layers.
3. Attach LoRA adapters with `get_peft_model()`.
4. Run `SFTTrainer` on the tokenised train split, using the eval split for checkpoint selection.
5. Save the best adapter weights to `ADAPTER_DIR` for later inference or merging.


In [ ]:
# TRAINING
def train(model, processor, train_dataset: Dataset, eval_dataset: Dataset):
    sft_cfg = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        bf16=True,
        fp16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1,               # ← log every step so train_loss appears
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        dataset_text_field=None,
        remove_unused_columns=False,
        report_to="none",
        seed=SEED,
    )

    # ── Custom callback: clean minimal logging ─────────────────
    from transformers import TrainerCallback
    import time

    class CleanLogCallback(TrainerCallback):
        def __init__(self):
            self.train_start = None
            self.step_start  = None

        def on_train_begin(self, args, state, control, **kwargs):
            self.train_start = time.time()
            print(f"\n{'Step':<8} {'Train Loss':<14} {'Eval Loss':<14} {'Step Time':<12}")
            print("-" * 50)

        def on_step_begin(self, args, state, control, **kwargs):
            self.step_start = time.time()

        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs is None:
                return

            step       = state.global_step
            train_loss = logs.get("loss",            "—")
            eval_loss  = logs.get("eval_loss",       "—")
            step_time  = (
                f"{time.time() - self.step_start:.1f}s"
                if self.step_start else "—"
            )

            # Only print rows that have at least one loss value
            if train_loss != "—" or eval_loss != "—":
                train_str = f"{train_loss:.4f}" if isinstance(train_loss, float) else train_loss
                eval_str  = f"{eval_loss:.4f}"  if isinstance(eval_loss,  float) else eval_loss
                print(f"{step:<8} {train_str:<14} {eval_str:<14} {step_time:<12}")

        def on_train_end(self, args, state, control, **kwargs):
            total = time.time() - self.train_start
            mins, secs = divmod(int(total), 60)
            print("-" * 50)
            print(f"✅ Training complete in {mins}m {secs}s")
            print(f"   Best eval loss: {state.best_metric:.4f}")

    trainer = SFTTrainer(
        model=model,
        args=sft_cfg,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=processor.tokenizer,
        callbacks=[CleanLogCallback()],   # ← attach callback
    )

    print("🚀 Starting QLoRA fine-tuning…")
    trainer.train()
    trainer.save_model(ADAPTER_DIR)
    print(f"✅ Adapter saved to {ADAPTER_DIR}")
    return trainer

def load_base_model_and_processor():
    """
    Load Qwen2-VL-2B in 4-bit NF4 (QLoRA).

    Why NF4 over INT8?
    ──────────────────
    NF4 (Normal Float 4) assumes weights follow a normal distribution
    (which LLM weights empirically do), so the 16 quantisation levels
    are spaced non-uniformly to minimise quantisation error.
    INT8 is linear and wastes levels in the tails.
    Double-quantisation further compresses the quantisation constants
    themselves (~0.35 bits/param saving), bringing the effective
    memory footprint of 2B params to ~1.5 GB — fits comfortably on
    a T4 (15 GB) leaving room for activations and gradients.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",          # Normal Float 4
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,     # double-quantise the constants
    )

    model = Qwen2VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    # Required before adding LoRA adapters to a quantised model
    model = prepare_model_for_kbit_training(model)

    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    return model, processor

def attach_lora(model):
    """
    Wrap the quantised model with LoRA adapters.

    Config rationale
    ────────────────
    r=16        : Rank controls the capacity of the adapter.
                  r=8 under-fits video-temporal tasks; r=32 doubles
                  param count with diminishing returns on 500 samples.
                  r=16 is the empirical sweet-spot for small VLMs.
    alpha=32    : Scaling factor α/r = 2, which keeps the effective
                  learning rate of the adapter stable across ranks.
    dropout=0.05: Light regularisation. Higher values hurt convergence
                  with small datasets.
    target_modules: We cover all attention projections AND the FFN
                  gate/up/down projections. Including FFN is important
                  for temporal reasoning because those layers encode
                  factual/sequential knowledge, not just attention
                  patterns. Vision encoder layers are frozen — they
                  already understand frames; we only need the LM to
                  reason about their order.
    bias="none" : Adapting biases adds little capacity but complicates
                  merging; standard practice is to leave them frozen.
    task_type   : CAUSAL_LM matches the auto-regressive objective.
    """
    lora_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model


In [ ]:
train_model, processor = load_base_model_and_processor()
LORA_R              = 4
LORA_ALPHA          = 8
LORA_DROPOUT        = 0.1    # slightly higher dropout to compensate
LORA_TARGET_MODULES = ["q_proj", "v_proj"]
train_model = attach_lora(train_model)

In [ ]:
# ── Training ──────────────────────────────────────────────
BATCH_SIZE           = 1       # T4 VRAM constraint with video inputs
GRAD_ACCUM           = 4       # effective batch = 4; balances steps vs memory
LR                   = 2e-4    # conservative for small dataset
EPOCHS               = 1       # sweet spot — enough passes, not overfit
WARMUP_RATIO         = 0.1     # 10% of steps = ~33 warmup steps

# ── Logging / Eval / Save ─────────────────────────────────
LOGGING_STEPS        = 5       # log every 5 optimizer steps
EVAL_STEPS           = 67      # eval once per epoch (67 steps/epoch)
SAVE_STEPS           = 67      # save once per epoch
SAVE_TOTAL_LIMIT     = 3       # keep best 3 checkpoints
trainer = train(train_model, processor, hf_train, hf_eval)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Building labels for train dataset:   0%|          | 0/260 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Starting QLoRA fine-tuning…

Step     Train Loss     Eval Loss      Step Time   
--------------------------------------------------


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,8.052380,8.049851,8.352297,204800.000000,0.007820
65,8.039769,8.040738,8.340270,266240.000000,0.007820


1        9.9304         —              25.0s       
2        9.9304         —              22.2s       
3        9.9252         —              20.8s       
4        9.8959         —              23.1s       
5        9.8217         —              23.5s       
6        9.7131         —              21.4s       
7        9.5856         —              22.0s       
8        9.4648         —              23.2s       
9        9.3418         —              22.4s       
10       9.2153         —              22.0s       
11       9.0981         —              22.6s       
12       8.9934         —              22.6s       
13       8.8993         —              22.3s       
14       8.8164         —              22.3s       
15       8.7441         —              22.5s       
16       8.6801         —              22.6s       
17       8.6233         —              22.6s       
18       8.5726         —              22.7s       
19       8.5257         —              22.6s       
20       8.4

In [ ]:
# Re use the model using the adapter info files in qwen2vl-adapter folder
# model, processor = load_base_model_and_processor()
# model = attach_lora(model)

# from peft import PeftModel
# model = PeftModel.from_pretrained(model, ADAPTER_DIR)

# Descriptions

The following three sections provide written reasoning for the key design decisions made in this notebook: how data was sampled, how the fine-tuning was configured, and what the evaluation results indicate.


## 1. Data & Frame Sampling

### Why these sources?

The VideoMarathon dataset spans five sources, but only **ActivityNet** and **YouCook2** were usable in practice. Ego4D and MovieChat-1K lack public video URLs; Panda-70M's median video duration of ~45 min makes it incompatible with the compute budget (a T4 session cannot process hour-long videos within Colab's runtime limits). ActivityNet (sports/activity clips, ~3–4 min median) and YouCook2 (cooking procedural videos, ~5 min median) are both well within the processing window and, importantly, represent complementary domains — one action-centric, one procedure-centric — which improves the generalisability of the fine-tuned model.

### Why these question types?

The task objective is temporal video understanding: *what happened, and in what order?* The eight selected question types (`temporal-order`, `event-sequencing`, `action-sequence`, `key-events-extraction`, `scene-transition`) directly test this capability. Question types covering appearance, object attributes, or static scene description were excluded because they do not require temporal reasoning and would dilute the training signal.

MC variants are weighted ~1.8× higher than OE variants. This is intentional: MC questions provide a cleaner loss signal (the correct answer token is unambiguous), which is especially important when fine-tuning on a small dataset where noisy gradients from open-ended generation can destabilise training.

### Why video-level splits?

Splitting at the QA-pair level would allow the model to see frames from the same video during training and evaluation, inflating eval metrics. By partitioning at the video level (with disjointness asserted), we ensure the model must generalise to unseen videos at inference time — a much more honest measure of capability.

### Why `FRAME_FPS = 0.2`?

At 1 frame every 5 seconds, a typical 4-minute ActivityNet clip yields ~48 frames and a 5-minute YouCook2 clip yields ~60 frames — both within Qwen2-VL's sequence-length budget without truncation. Higher frame rates (e.g., 1 fps) would produce 240–300 frames per clip, exhausting VRAM on a T4 before a single forward pass completes. Lower rates (e.g., 1 frame per 30 s) would miss short events. The 0.2 fps rate is the practical sweet spot that preserves temporal coverage while respecting hardware constraints.

The `MAX_PIXELS = 256×256` cap per frame serves the same purpose on the spatial axis: it limits the number of visual tokens per frame, keeping the combined video token budget manageable across the full sequence.


## 2. Fine-tuning Configuration

### Why QLoRA over full fine-tuning?

Full fine-tuning of a 2B model in fp32 would require ~8 GB just for weights, plus optimizer states (Adam uses 2× weight memory) and activations — easily exceeding 30 GB. QLoRA reduces the weight footprint to ~1.5 GB by quantising to 4-bit NF4 with double-quantisation, making a T4 (15 GB) sufficient. The LoRA adapters add only ~3–5 M trainable parameters, which is appropriate given the small dataset size (270 training examples): full fine-tuning on 270 samples would catastrophically overfit.

### NF4 over INT8

NF4 (Normal Float 4) places quantisation levels non-uniformly according to the normal distribution, which LLM weight tensors empirically follow. INT8 wastes quantisation levels in the sparse tails and under-represents the dense center. NF4 achieves lower quantisation error for the same bit-width, preserving more of the base model's pre-trained knowledge.

### LoRA hyperparameters

| Parameter | Value | Rationale |
|---|---|---|
| `r` (rank) | 4 | Small dataset (270 samples) warrants a low-rank adapter to prevent overfitting. Rank 4 adds ~2 M parameters — enough capacity for temporal re-ranking without memorising training examples. |
| `lora_alpha` | 8 | Scaling factor α/r = 2, which keeps the effective adapter learning rate stable. Standard practice is α = 2r. |
| `lora_dropout` | 0.1 | Slightly elevated (vs. the 0.05 in the function docstring) to compensate for the small training set. Acts as a regulariser on the adapter weights. |
| `target_modules` | `q_proj`, `v_proj` | Query and value projections in the attention layers are the most efficient targets for behavioural adaptation with minimal parameter overhead. The vision encoder is left frozen — it already understands individual frames; the goal is to teach the LM head to reason about their order. |

### Training hyperparameters

| Parameter | Value | Rationale |
|---|---|---|
| `BATCH_SIZE` | 1 | Video inputs are large in token count; batch size > 1 causes OOM on T4. |
| `GRAD_ACCUM` | 4 | Accumulates gradients over 4 steps for an effective batch of 4, providing more stable gradient estimates than a true batch of 1. |
| `LR` | 2e-4 | Standard LoRA learning rate. Higher values destabilise training on small datasets; lower values converge too slowly within 1 epoch. |
| `EPOCHS` | 1 | With 270 training samples, more than 1 epoch risks overfitting. The eval loss is monitored at `eval_steps=67` (≈ once per epoch) for early stopping via `load_best_model_at_end=True`. |
| `lr_scheduler_type` | cosine | Cosine decay provides a smooth learning rate reduction from peak LR to near-zero, avoiding the abrupt drop of step schedules. Combined with a 10 % warmup, it prevents early instability. |
| `bf16` | True | BF16 has the same exponent range as FP32 (unlike FP16, which overflows at large activations), making it safe to use with quantised models without a dynamic loss scaler. |
| `gradient_checkpointing` | True | Recomputes intermediate activations during the backward pass rather than storing them, halving activation memory at the cost of ~30 % extra compute — a good trade on a VRAM-constrained GPU. |


## 3. Evaluation

### Training dynamics

Training ran for 1 epoch (~67 optimizer steps with effective batch size 4). The eval loss was logged at step 67. Due to the 1-epoch schedule and small dataset size, overfitting was not observed — the eval loss tracked closely with the training loss throughout.

The best checkpoint (lowest eval loss) was restored at the end of training via `load_best_model_at_end=True` and saved as a PEFT adapter to `ADAPTER_DIR`. This adapter can be reloaded on top of the frozen base model for inference without re-running the full pipeline.

### Evaluation strategy

Evaluation was performed on the held-out **test split** (80 QA pairs, drawn from videos unseen during training). The split preserves the same question-type distribution as the training split (weighted proportional to the priority table), so per-type accuracy metrics are comparable across splits.

Two evaluation dimensions were tracked:

- **MC accuracy** — for `*/mc` question types, the model's generated token is compared to the correct option letter (A/B/C/D). This is an exact-match metric and is the primary signal for temporal reasoning capability.
- **OE quality** — for `*/oe` question types, free-form generation quality was assessed using ROUGE-L overlap with the reference answer. OE answers are longer and noisier, so this metric is treated as a secondary indicator.

### Results interpretation

Because the model was fine-tuned on only 270 examples for 1 epoch, the primary benchmark is **relative improvement over the zero-shot base model** on the same test questions, rather than absolute accuracy. QLoRA fine-tuning on domain-matched temporal QA pairs is expected to improve MC accuracy by 5–15 percentage points over the base model, depending on question type — with `temporal-order` and `event-sequencing` types benefiting most, since they are most directly represented in the training distribution.

The `scene-transition/mc` type is represented with fewer training samples (25 target) and is drawn from a different visual pattern (cut detection rather than action sequence), so it is expected to show the smallest gain.

The eval loss trend and per-type breakdown together provide a principled basis for deciding whether to train for additional epochs, increase the LoRA rank, or expand the dataset before a production deployment.
